# Blasius Boundary Layer

This notebook compares MI-DL, SI-DL, and the known Blasius similarity coordinate for the laminar boundary-layer velocity profile. It generates the numerical Blasius solution, searches for dimensionless coordinates, and compares their information and covariance scores.


## 1. Imports and settings

Configure the Blasius solver, dimensional matrix, MI-DL restarts, and SI-DL optimization. Results are written to `csv/` and the two retained figures to `figures/`.


In [ ]:
from __future__ import annotations

import os

import sys

import warnings

from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/matplotlib-codex-cache")

os.environ.setdefault("MPLBACKEND", "Agg")

warnings.filterwarnings("ignore", category=RuntimeWarning)

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from scipy.integrate import odeint

from scipy.optimize import differential_evolution, root

OUTPUT_DIR = Path.cwd()

ROOT = OUTPUT_DIR.parent

for module_dir in [ROOT / "Compare" / "MI-DL", ROOT / "SI-DL-main"]:
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

import midl

import SI_DL

RANDOM_STATE = 43

FIG_DIR = OUTPUT_DIR / "figures"

CSV_DIR = OUTPUT_DIR / "csv"

INPUT_COLUMNS = ["U", "nu", "x", "y"]

VARIABLE_LABELS = ["U", "nu", "x", "y"]

MI_K_NEIGHBORS = 6

MI_DE_MAXITER = 180

MI_RESTART_SEEDS = [0, 42, 101]

SI_BANDWIDTH = 0.01

SI_MAXITER = 70

SI_POPSIZE = 8

D_IN = np.array(
    [
        [1, 2, 1, 1],
        [-1, -1, 0, 0],
    ],
    dtype=float,
)

PHYSICAL_BASIS = np.array(
    [
        [1.0, -1.0, 1.0, 0.0],
        [0.0, 0.0, -1.0, 1.0],
    ],
    dtype=float,
)

KNOWN_BLASIUS = np.array([0.5, -0.5, -0.5, 1.0], dtype=float)


## 2. Generate and trim the Blasius dataset

Solve the Blasius initial-value problem, construct physical samples, and remove boundary points that would dominate the comparison.


In [ ]:
def blasius_rhs(f: np.ndarray) -> np.ndarray:
    return np.array([f[1], f[2], -0.5 * f[0] * f[2]])

def solve_blasius_initial_condition() -> list[float]:
    eta_grid = np.arange(0.0, 10.0, 0.01)

    def boundary_residual(f0: np.ndarray) -> list[float]:
        f = odeint(lambda state, _eta: blasius_rhs(state), f0, eta_grid)
        return [f0[0], f0[1], f[-1, 1] - 1.0]

    opt = root(boundary_residual, [0.0, 0.0, 0.0], tol=1e-4)
    return [0.0, 0.0, float(opt.x[2])]

def generate_blasius_data(n_x: int = 20, n_y: int = 20) -> pd.DataFrame:
    U_inf = 0.01
    nu = 1e-6

    x = np.linspace(1e-3, 1e-1, n_x)
    reynolds = (U_inf / nu) * x

    delta = 1.72 * np.sqrt(x * nu / U_inf)
    y = np.linspace(1e-4, 2.0 * float(np.max(delta)), n_y)

    yy, xx = np.meshgrid(y, x)
    eta = yy * np.sqrt(U_inf / (xx * nu))

    f0 = solve_blasius_initial_condition()

    u = np.zeros_like(eta)
    v = np.zeros_like(eta)

    for i in range(len(x)):
        f = odeint(lambda state, _eta: blasius_rhs(state), f0, eta[i, :])
        u[i, :] = U_inf * f[:, 1]
        v[i, :] = (0.5 * U_inf / np.sqrt(reynolds[i])) * (
            eta[i, :] * f[:, 1] - f[:, 0]
        )

    return pd.DataFrame(
        {
            "U": np.full(u.size, U_inf),
            "nu": np.full(u.size, nu),
            "x": xx.ravel(),
            "y": yy.ravel(),
            "u": u.ravel(),
            "v": v.ravel(),
            "u_over_U": (u / U_inf).ravel(),
            "v_over_U": (v / U_inf).ravel(),
            "known_eta": eta.ravel(),
        }
    )

def trim_boundary_points(df: pd.DataFrame) -> pd.DataFrame:
    x_min, x_max = df["x"].min(), df["x"].max()
    y_min, y_max = df["y"].min(), df["y"].max()

    return df[
        (df["x"] > x_min)
        & (df["x"] < x_max)
        & (df["y"] > y_min)
        & (df["y"] < y_max)
    ].copy()


## 3. Dimensionless-coordinate utilities

Normalize exponent vectors, construct candidate coordinates, format formulas, and compute common information metrics.


In [ ]:
def normalize_exponents(exponents: np.ndarray) -> np.ndarray:
    row = np.asarray(exponents, dtype=float).reshape(-1)
    scale = float(np.max(np.abs(row)))

    if scale <= 1e-12:
        return row

    out = row / scale
    first = np.flatnonzero(np.abs(out) > 1e-10)

    if first.size and out[first[0]] < 0.0:
        out = -out

    return out

def pi_from_exponents(X: np.ndarray, exponents: np.ndarray) -> np.ndarray:
    return np.exp(
        np.log(np.asarray(X, dtype=float))
        @ np.asarray(exponents, dtype=float).reshape(-1)
    )

def formula_from_exponents(exponents: np.ndarray, decimals: int = 4) -> str:
    terms = []

    for label, value in zip(VARIABLE_LABELS, np.asarray(exponents).reshape(-1)):
        value = float(value)
        if abs(value) < 5e-5:
            continue
        terms.append(f"{label}^{value:.{decimals}f}")

    return " * ".join(terms)

def information_metrics(feature: np.ndarray, Y: np.ndarray) -> dict:
    info = midl.information_lower_bound(
        np.asarray(feature, dtype=float).reshape(-1, 1),
        Y,
        k_neighbors=MI_K_NEIGHBORS,
        random_state=RANDOM_STATE,
    )

    return {
        "mutual_information": float(info["mutual_information"]),
        "epsilon_lb_normalized": float(info["epsilon_lb"] / np.var(Y, ddof=0)),
    }


## 4. MI-DL and SI-DL searches

Run the information-based and covariance-based optimizations in the dimensional null space.


In [ ]:
def run_midl_search(X: np.ndarray, Y: np.ndarray) -> dict:
    basis, _ = midl.calc_basis(D_IN)
    log_pi_basis = np.log(X) @ basis

    rows = []
    best = None

    for seed in MI_RESTART_SEEDS:
        model = midl.MIDL(
            k_neighbors=MI_K_NEIGHBORS,
            de_maxiter=MI_DE_MAXITER,
            random_state=seed,
        )

        w, _search_mi, _ = model._optimize_direction_in_subspace(
            log_pi_basis,
            Y,
            np.eye(basis.shape[1]),
        )

        exponents = normalize_exponents(np.asarray(basis @ w).reshape(-1))
        pi = pi_from_exponents(X, exponents)
        mi = information_metrics(pi, Y)

        row = {
            "seed": seed,
            "mi_raw_pi": mi["mutual_information"],
            "epsilon_lb_raw_pi_normalized": mi["epsilon_lb_normalized"],
            "formula": formula_from_exponents(exponents),
            "exponents": exponents,
        }

        rows.append(row)

        if best is None or row["mi_raw_pi"] > best["mi_raw_pi"]:
            best = row

    audit = pd.DataFrame(
        [{k: v for k, v in row.items() if k != "exponents"} for row in rows]
    )
    audit = audit.sort_values("mi_raw_pi", ascending=False)

    return {
        "exponents": best["exponents"],
        "best_seed": int(best["seed"]),
        "restart_audit": audit,
    }

def params_to_exponents(params: np.ndarray) -> np.ndarray:
    return normalize_exponents(
        np.asarray(params, dtype=float).reshape(-1) @ PHYSICAL_BASIS
    )

def run_sidl_search(X: np.ndarray, Y: np.ndarray) -> dict:
    def objective(params: np.ndarray) -> float:
        try:
            pi = pi_from_exponents(X, params_to_exponents(params))
            score = SI_DL.explained_variance_score(
                pi,
                Y,
                bandwidth=SI_BANDWIDTH,
            )["S_cov"]
        except Exception:
            return 1e6

        if not np.isfinite(score):
            return 1e6

        return -float(score)

    result = differential_evolution(
        objective,
        bounds=[(-2.0, 2.0)] * PHYSICAL_BASIS.shape[0],
        maxiter=SI_MAXITER,
        popsize=SI_POPSIZE,
        seed=RANDOM_STATE,
        polish=True,
        updating="immediate",
        workers=1,
    )

    exponents = params_to_exponents(result.x)

    return {
        "exponents": exponents,
        "optimizer_result": result,
    }


## 5. Scores and retained figures

Evaluate all coordinates, draw the raw-coordinate comparison, and render the compact summary table.


In [ ]:
def score_metrics(feature: np.ndarray, Y: np.ndarray) -> dict:
    sidl_score = SI_DL.explained_variance_score(
        feature,
        Y,
        bandwidth=SI_BANDWIDTH,
    )

    mi = information_metrics(feature, Y)

    return {
        **mi,
        "S_cov": float(sidl_score["S_cov"]),
        "sidl_error": float(1.0 - sidl_score["S_cov"]),
        "sidl_bandwidth": float(sidl_score["bandwidth"]),
        "sidl_n_retained": int(sidl_score["n_retained"]),
    }

def plot_pi_scatter(coordinates: dict, Y: np.ndarray, summary: pd.DataFrame) -> None:
    methods = (
        ("MI-DL", r"MI-DL $\mathit{\Pi}_{1}$"),
        ("SI-DL", r"SI-DL $\mathit{\Pi}_{1}$"),
        ("Known Blasius", r"Known Blasius $\mathit{\Pi}_{1}$"),
    )

    plt.rcParams.update(
        {
            "font.family": "STIXGeneral",
            "mathtext.fontset": "stix",
            "axes.titlesize": 22,
            "axes.labelsize": 21,
            "xtick.labelsize": 17,
            "ytick.labelsize": 17,
            "legend.fontsize": 15,
        }
    )

    fig, axes = plt.subplots(1, 3, figsize=(17.2, 5.7), sharey=True, dpi=260)

    for ax, (method, xlabel) in zip(axes, methods):
        feature = coordinates[method]["pi"]

        ax.scatter(
            feature,
            Y,
            s=42,
            color="#1f77b4",
            alpha=0.58,
            edgecolors="white",
            linewidths=0.45,
            label="data",
            zorder=2,
        )

        ax.set_xlabel(xlabel)
        ax.set_title(method)
        ax.grid(False)
        ax.legend(
            frameon=True,
            facecolor="white",
            edgecolor="#d1d5db",
            framealpha=0.94,
            loc="upper left",
            handlelength=1.4,
            borderpad=0.65,
        )

    axes[0].set_ylabel(r"$u/U$")

    fig.suptitle(
        "Blasius Boundary Layer",
        fontsize=25,
        y=1.02,
    )

    fig.tight_layout()

    fig.savefig(
        FIG_DIR / "comparison.png",
        dpi=260,
        bbox_inches="tight",
    )

    plt.close(fig)

def plot_summary_table(summary: pd.DataFrame) -> None:
    fig, ax = plt.subplots(figsize=(14.5, 4.8), dpi=220)
    ax.axis("off")
    fig.patch.set_facecolor("white")

    ax.text(
        0.5,
        0.96,
        "Blasius comparison, k=6",
        ha="center",
        va="top",
        fontsize=18,
        weight="bold",
    )

    rows = []

    for _, row in summary.iterrows():
        rows.append(
            [
                row["method"],
                row["formula"],
                f"{row['mutual_information']:.4f}",
                f"{row['epsilon_lb_normalized']:.5f}",
                f"{row['S_cov']:.4f}",
                f"{row['sidl_error']:.5f}",
            ]
        )

    table = ax.table(
        cellText=rows,
        colLabels=[
            "Method",
            "Found pi",
            "MI k=6",
            "epsilon_LB/Var",
            "S_cov",
            "1 - S_cov",
        ],
        cellLoc="center",
        colLoc="center",
        colWidths=[0.13, 0.50, 0.09, 0.11, 0.08, 0.09],
        bbox=[0.02, 0.08, 0.96, 0.78],
    )

    table.auto_set_font_size(False)

    for (r, c), cell in table.get_celld().items():
        cell.set_edgecolor("#3f3f46")
        cell.set_linewidth(0.85)
        cell.PAD = 0.03

        text = cell.get_text()

        if r == 0:
            cell.set_facecolor("#1f2937")
            text.set_color("white")
            text.set_weight("bold")
            text.set_fontsize(10.5)
        else:
            cell.set_facecolor("#f8fafc" if r % 2 == 1 else "#ffffff")
            text.set_fontsize(8.2 if c == 1 else 10.0)

            if c == 1:
                text.set_ha("left")

            if c == 0:
                text.set_weight("bold")

    fig.savefig(
        FIG_DIR / "summary.png",
        bbox_inches="tight",
        facecolor="white",
    )

    plt.close(fig)


## 6. Run the comparison

Generate the data, execute both searches, save four CSV result tables, and create exactly two figures.


In [ ]:
def run_comparison() -> dict:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    CSV_DIR.mkdir(parents=True, exist_ok=True)

    df = generate_blasius_data(n_x=30, n_y=30)

    # Remove boundary points. Comment this line out if you want to keep all points.
    df = trim_boundary_points(df)

    X = df[INPUT_COLUMNS].to_numpy(float)
    Y_clean = df["u_over_U"].to_numpy(float)

    noise_level = 0.05
    rng = np.random.default_rng(RANDOM_STATE)
    noise = noise_level * np.std(Y_clean) * rng.normal(size=Y_clean.shape)

    Y = Y_clean + noise

    df["u_over_U_clean"] = Y_clean
    df["u_over_U_noisy"] = Y
    df["noise"] = noise

    midl = run_midl_search(X, Y)
    sidl = run_sidl_search(X, Y)

    coordinates = {
        "MI-DL": {"exponents": midl["exponents"]},
        "SI-DL": {"exponents": sidl["exponents"]},
        "Known Blasius": {"exponents": KNOWN_BLASIUS},
    }

    for values in coordinates.values():
        values["pi"] = pi_from_exponents(X, values["exponents"])

    rows = []

    for method, values in coordinates.items():
        common = score_metrics(values["pi"], Y)

        rows.append(
            {
                "method": method,
                "formula": formula_from_exponents(values["exponents"]),
                "exponents": values["exponents"],
                "n_samples": int(Y.size),
                "feature_space": "raw_pi",
                **common,
                "best_seed": midl["best_seed"] if method == "MI-DL" else np.nan,
            }
        )

    summary = pd.DataFrame(rows)

    summary["rank_by_MI"] = summary["mutual_information"].rank(
        ascending=False,
        method="min",
    ).astype(int)

    summary["rank_by_S_cov"] = summary["S_cov"].rank(
        ascending=False,
        method="min",
    ).astype(int)

    summary_csv = summary.copy()

    for j, label in enumerate(VARIABLE_LABELS):
        summary_csv[f"pi1_exp_{label}"] = summary_csv["exponents"].map(
            lambda arr, jj=j: arr[jj]
        )

    summary_csv.drop(columns=["exponents"]).to_csv(
        CSV_DIR / "summary.csv",
        index=False,
    )

    midl["restart_audit"].to_csv(
        CSV_DIR / "midl_restarts.csv",
        index=False,
    )

    coord_out = df.copy()

    for method, values in coordinates.items():
        coord_out[f"{method}_pi1"] = values["pi"]

    coord_out.to_csv(
        CSV_DIR / "coordinates.csv",
        index=False,
    )

    exponent_rows = []

    for method, values in coordinates.items():
        for label, exponent in zip(VARIABLE_LABELS, values["exponents"]):
            exponent_rows.append(
                {
                    "method": method,
                    "variable": label,
                    "normalized_exponent": float(exponent),
                }
            )

    pd.DataFrame(exponent_rows).to_csv(
        CSV_DIR / "exponents.csv",
        index=False,
    )

    plot_pi_scatter(coordinates, Y, summary)
    plot_summary_table(summary)

    return {"summary": summary}


In [ ]:
outputs = run_comparison()
outputs["summary"]
